# 📘 Stateful Chatbot with Conversation History — Full Explanation

# 🔶 1. 📦 Setup & Load Environment Variables

### 🎯 Purpose

Load secret API keys securely and prepare the project environment.

---

## 🔑 Key Functions

```python
from dotenv import load_dotenv
import os
```

```python
load_dotenv()
```

### ✅ What it does

Loads variables from a `.env` file into Python.

Example `.env` file:

```env
GROQ_API_KEY=abc123
OPENAI_API_KEY=xyz456
```

---

```python
os.getenv("GROQ_API_KEY")
```

### ✅ What it does

Fetches API keys securely from environment variables.

---

# 💡 Why This Is Important

❌ Bad Practice:

```python
api_key = "my_secret_key"
```

✅ Good Practice:

```python
api_key = os.getenv("GROQ_API_KEY")
```

This keeps secrets secure and production-ready.

---

# 🔶 2. 🤖 Initialize the LLM (Large Language Model)

### 🎯 Purpose

Connect your application to an AI model.

---

## 🔑 Key Functions

```python
from langchain_groq import ChatGroq
```

```python
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="Llama3-8b-8192"
)
```

---

# 💡 What Is Happening Here?

You are creating an object that talks to Groq servers.

Internally:

```text
Your App → Groq API → Llama Model → Response
```

---

# 💡 Why Use ChatGroq?

Groq provides:

* Very fast inference
* Open-source models
* Low latency

You could replace this with:

* ChatOpenAI
* Ollama
* Gemini
* HuggingFace

Same architecture.

---

# 🔶 3. 🌐 Load Data from Website

### 🎯 Purpose

Collect external knowledge for the chatbot.

---

## 🔑 Key Functions

```python
from langchain_community.document_loaders import WebBaseLoader
```

```python
loader = WebBaseLoader(web_paths=("URL",))
docs = loader.load()
```

---

# 💡 What Is Happening?

The chatbot itself does NOT know your custom data.

So we:

1. Scrape webpage content
2. Convert into documents
3. Feed into retrieval pipeline

---

# 💡 Example

Suppose website contains:

```text
Machine learning is a subset of AI...
```

The loader extracts that text.

---

# 🔶 4. ✂️ Split Text into Chunks

### 🎯 Purpose

Break large text into smaller manageable pieces.

---

## 🔑 Key Functions

```python
from langchain.text_splitter import RecursiveCharacterTextSplitter
```

```python
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
```

---

# 💡 Why Splitting Is Needed

LLMs cannot process huge documents efficiently.

So instead of:

```text
Entire 100-page document
```

we create:

```text
Chunk 1
Chunk 2
Chunk 3
```

---

# 💡 chunk_overlap

Overlap preserves context between chunks.

Without overlap:

* sentences may break
* meaning may be lost

---

# 🔶 5. 🧠 Create Embeddings

### 🎯 Purpose

Convert text into numerical vectors.

---

## 🔑 Key Functions

```python
from langchain_huggingface import HuggingFaceEmbeddings
```

```python
embeddings = HuggingFaceEmbeddings()
```

---

# 💡 What Are Embeddings?

Embeddings convert text into numbers.

Example:

```text
"dog" → [0.23, 0.91, 0.77...]
```

LLMs cannot understand raw text directly for similarity search.

Vectors help compare meanings.

---

# 💡 Why Important?

This enables:
✅ semantic search
✅ similarity matching
✅ retrieval systems

---

# 🔶 6. 🗂️ Store Embeddings in Vector Database

### 🎯 Purpose

Store vectors for efficient retrieval.

---

## 🔑 Key Functions

```python
from langchain_community.vectorstores import Chroma
```

```python
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)
```

---

# 💡 What Happens Here?

Each chunk becomes:

```text
Chunk → Embedding Vector → Stored in DB
```

---

# 💡 Why Vector DB?

When user asks question:

```text
"What is AI?"
```

System searches:
👉 “Which chunks are most similar?”

---

# 🔶 7. 🔍 Create Retriever

### 🎯 Purpose

Fetch relevant chunks from vector database.

---

## 🔑 Key Functions

```python
retriever = vectorstore.as_retriever()
```

---

# 💡 What Retriever Does

User Question:

```text
"What is machine learning?"
```

Retriever searches vector DB.

Returns:

```text
Relevant chunk about machine learning
```

---

# 🔶 8. 🧾 Prompt Template

### 🎯 Purpose

Control how the LLM behaves.

---

## 🔑 Key Functions

```python
ChatPromptTemplate
```

```python
MessagesPlaceholder
```

---

# 💡 System Prompt

```python
"You are an assistant..."
```

This controls:

* tone
* response style
* constraints
* behavior

---

# 💡 MessagesPlaceholder

```python
MessagesPlaceholder("chat_history")
```

This inserts previous conversations dynamically.

VERY important for memory.

---

# 🔶 9. 🧠 History-Aware Retriever

### 🎯 Purpose

Understand follow-up questions using conversation history.

---

# 💡 Problem Without This

User asks:

```text
What is LangChain?
```

Then:

```text
Who created it?
```

Retriever gets confused:
👉 “Who created WHAT?”

---

# 💡 Solution

Use chat history.

System reformulates question into:

```text
Who created LangChain?
```

Now retrieval becomes accurate.

---

# 🔶 10. ⛓️ Create RAG Chain

### 🎯 Purpose

Combine retrieval + prompting + LLM.

---

# 💡 Flow

```text
User Question
→ Retrieve Relevant Chunks
→ Insert into Prompt
→ Send to LLM
→ Generate Answer
```

This is called:

# 🌟 Retrieval-Augmented Generation (RAG)

---

# 🔶 11. 💬 Add Conversation Memory

### 🎯 Purpose

Make chatbot stateful.

---

## 🔑 Key Components

```python
ChatMessageHistory
RunnableWithMessageHistory
```

---

# 💡 What Happens?

Without memory:

```text
Every request is independent
```

With memory:

```text
Bot remembers previous chats
```

---

# 🔶 12. 🪪 Session-Based Chat History

### 🎯 Purpose

Support multiple users independently.

---

## 🔑 Key Function

```python
get_session_history(session_id)
```

---

# 💡 Why Important?

Without sessions:

❌ All users share same memory

With sessions:

✅ Every user gets separate conversation history

Exactly like ChatGPT.

---

# 🔶 13. ⚡ RunnableWithMessageHistory

### 🎯 Purpose

Automatically manage chat history.

---

# 💡 Internal Flow

```text
User Message
→ Load History
→ Add to Prompt
→ Send to LLM
→ Get Response
→ Save New Messages
```

This wrapper automates everything.

---

# 🔶 14. 🧹 Token & Message Trimming

### 🎯 Purpose

Prevent context overflow.

---

## 🔑 Key Function

```python
trim_messages()
```

---

# 💡 Why Needed?

Conversations grow forever.

LLMs have context limits.

Example:

```text
8K tokens
32K tokens
128K tokens
```

Too much history:
❌ expensive
❌ slower
❌ crashes context window

---

# 💡 Solution

Keep:
✅ recent messages
✅ important messages

Remove:
❌ unnecessary old chats

---

# 🔶 15. 🚀 Final Architecture

# 🌟 Complete Chatbot Pipeline

```text
User Input
↓
Load Session History
↓
Retrieve Relevant Documents
↓
Trim Messages
↓
Insert into Prompt
↓
Send to LLM
↓
Generate Response
↓
Store Back into Memory
```
